[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [scikit-learn, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/scikit-learn-deep-dive.html)

# Your First Model


## What you will be able to do

Hold some stations back, train a model on the rest, and score it on the stations it never saw, in
six lines that work the same way for every model in scikit-learn.

## The idea

### The problem

**The Estimator API** notebook fitted three models of January temperature on all 1,200 stations and
scored them on the same 1,200. The random forest came out at 0.984, close to perfect. That number
measures how well a model remembers stations it has already been shown. The question that matters is
the planned station on the northern hillside, which no model has seen, and a score on familiar rows
says little about it.

A model can do well on its training rows in two very different ways. It can find the pattern that
links place to temperature, which carries over to any new station. Or it can memorize the training
stations one by one, noise and all, which carries over to nothing. Both look the same on the rows it
learned from, and they come apart only on rows it has not seen.

So the habit to build from the first model onward is to set some stations aside before training,
train on the others, and judge the model only by the stations it was never given.

### What a held-out score is

> A **test set** is a part of the data kept away from the model while it trains. The rest is the
> **training set**. A model's **score on the test set** estimates how well it will do on new data of
> the same kind, because the test rows played no part in fitting it. `train_test_split` divides
> `X` and `y` at random into the two parts, and `score(X_test, y_test)` measures the model on the
> held-out part.

### Why it works that way

The test set stands in for the future. Whatever the model learned from the training stations, it
has to apply it to stations it knows nothing about, which is what will happen to the planned
station too. If its score holds up on the test set, the pattern it found was real. If the score
falls, the model learned something about the training stations in particular.

`score` means something different for different kinds of model, and both meanings appear here. A
regressor, which predicts a number such as a January mean, reports **R²**, the share of the
variation in the target that its predictions account for, where 1.0 is perfect and 0.0 is no better
than always predicting the mean. A classifier, which predicts a category such as snow or no snow,
reports **accuracy**, the share of rows it gets right.

The split is random, so the score depends a little on which stations ended up in the test set. A
fixed `random_state` makes the split the same on every run, so the numbers below come out the same
for you as they did here. The **Train and Test** notebook looks at how much the score moves when the
seed changes.

### Where this shows up

Every evaluation in machine learning starts with a split like this one, and every notebook that
follows in this guide splits before it fits. The **Cross-Validation** notebook repeats the split
several times to use every station for testing once, and the **Pipelines** notebook keeps
preparation steps from seeing the test rows. The Visual Handbook covers the statistics of fitting
and overfitting, and this notebook covers the scikit-learn calls.

### What this notebook covers

- the six lines that split, train, predict and score
- predictions for stations the model has not seen, and how far off they are
- what R² measures, worked out by hand
- three models scored on the same held-out stations
- accuracy for a classifier, against the score of always guessing the more common answer
- three errors, among them a perfect score that means nothing

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read
it, and read the output underneath it. Everything from Setup onward is where you start running
things.

```python
X = network[FEATURES]
y = network["jan_mean"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=0)
model = LinearRegression()
model.fit(X_train, y_train)
print(round(model.score(X_test, y_test), 3))
```

```
0.888
```

`train_test_split` kept a quarter of the stations back. The model learned from the other three
quarters, and its R² on the stations it never saw is 0.888, so its predictions account for most of
the variation in January temperature across those stations. The lines use `network` and `FEATURES`,
which Setup makes.

## Setup

Two libraries for the data, scikit-learn, and five of its tools.

- `numpy` and `pandas` build the stations, and the cell prints scikit-learn's version
- `train_test_split` divides the stations into a training set and a test set
- `LinearRegression`, `DecisionTreeRegressor` and `RandomForestRegressor` are three models of
  January temperature, scored side by side
- `DecisionTreeClassifier` predicts whether a station has snow on 1 May, for accuracy

`station_network` makes the same network of 1,200 weather stations in Norway as **The Estimator
API** notebook. The stations are made up, and a generator started from a fixed seed draws them, so
the table is the same on every run. `FEATURES` names the five numeric columns the models use.

In [1]:
import numpy as np
import pandas as pd
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

def station_network(n=1200, seed=2025):
    """A made-up network of weather stations in Norway, one row each, with three things to predict."""
    rng = np.random.default_rng(seed)
    latitude = rng.uniform(58.0, 71.0, n)
    longitude = 5 + 1.5 * (latitude - 58) + rng.uniform(0, 5, n)
    coast_km = np.minimum(rng.exponential(25, n), 250)
    elevation = np.clip(rng.gamma(1.5, 150, n) + 3 * coast_km, 0, 1800)
    terrain = np.where(coast_km < 3, "coastal",
              np.where(elevation > 700, "mountain",
              np.where((coast_km < 30) & (rng.uniform(size=n) < 0.6), "fjord", "inland")))
    owner = rng.choice(["met", "road", "airport", "private"], n, p=[0.4, 0.3, 0.1, 0.2])
    installed = rng.integers(1950, 2025, n)
    shift = pd.Series(terrain).map({"coastal": 0.8, "fjord": 0.3, "inland": 0.0, "mountain": -0.5}).to_numpy()
    jan_mean = (1.5 - 0.55 * (latitude - 58) - 0.006 * elevation - 4.0 * (1 - np.exp(-coast_km / 40))
                + shift + rng.normal(0, 0.9, n))
    may_mean = jan_mean + 8.5 + 0.004 * coast_km + rng.normal(0, 1.2, n)
    june_min = 8.5 - 0.35 * (latitude - 58) - 0.0075 * elevation - 0.02 * coast_km + rng.normal(0, 1.5, n)
    table = pd.DataFrame({
        "station": [f"ST{i:04d}" for i in range(1, n + 1)],
        "latitude": latitude.round(2), "longitude": longitude.round(2),
        "elevation_m": elevation.round(0).astype(int), "coast_km": coast_km.round(1),
        "terrain": terrain, "owner": owner, "installed": installed,
        "jan_mean": jan_mean.round(1),
        "snow_on_may_1": may_mean < 2.5,
        "june_frost": june_min < -0.8,
    })
    return table


network = station_network()
FEATURES = ["latitude", "longitude", "elevation_m", "coast_km", "installed"]

print("scikit-learn", sklearn.__version__, "|", len(network), "stations")

scikit-learn 1.6.1 | 1200 stations


## Worked examples

### Holding stations back

`train_test_split` takes `X` and `y` together and returns four pieces: the training and test parts
of `X`, then the training and test parts of `y`, in that order. `test_size=0.25` keeps a quarter for
testing:

In [2]:
X = network[FEATURES]
y = network["jan_mean"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=0)

print("training:", X_train.shape, y_train.shape)
print("test:    ", X_test.shape, y_test.shape)
print("first test stations:", X_test.index[:3].tolist())

training: (900, 5) (900,)
test:     (300, 5) (300,)
first test stations: [198, 328, 45]


900 stations to learn from and 300 to test on. The split shuffled the rows first, so the test set
starts with the rows labeled 198, 328 and 45 in the table's index, and each part of `y` holds the
answers for the same stations as the matching part of `X`.

### Training on the training set

In [3]:
model = LinearRegression()
model.fit(X_train, y_train)

print("predicted:", model.predict(X_test.head(3)).round(1))
print("measured: ", y_test.head(3).to_numpy())

predicted: [-4.5 -7.4 -5.5]
measured:  [-5.1 -7.8 -4.4]


These three are stations the model has never seen, so this time the comparison means something. The
first prediction is 0.6 degrees too warm, the second 0.4 too warm and the third 1.1 too cold.

### How far off the predictions are across the test set

In [4]:
errors = model.predict(X_test) - y_test

print("mean error, ignoring its sign:", round(errors.abs().mean(), 2))
print("largest error:", round(errors.abs().max(), 2))
print("spread of the measured values:", round(y_test.std(), 2))

mean error, ignoring its sign: 0.78
largest error: 3.28
spread of the measured values: 3.0


On average a prediction is 0.78 degrees from the measured January mean, and the worst is 3.28
degrees off. The measured means themselves have a standard deviation of 3.0 degrees across these
stations, so the model has explained most of what separates one station from another. The
**Regression Metrics** notebook turns errors like these into the standard measures.

### What score means for a regressor

In [5]:
print("R² on the test set:    ", round(model.score(X_test, y_test), 3))
print("R² on the training set:", round(model.score(X_train, y_train), 3))

R² on the test set:     0.888
R² on the training set: 0.879


A linear model has only six numbers to learn, so it cannot memorize 900 stations, and its scores on
the two sets are close. The test score is slightly the higher here, which happens when the test
stations are a little easier than the training ones. R² compares the model's squared errors with
those of always predicting the mean, and working it out by hand gives the same number as `score`:

In [6]:
squared_errors = (y_test - model.predict(X_test)) ** 2
squared_spread = (y_test - y_test.mean()) ** 2

print(round(1 - squared_errors.sum() / squared_spread.sum(), 3))

0.888


The model's squared errors are about a ninth of the squared distances from the mean, so R² is
about 0.89.

### Comparing models on the same held-out stations

The six lines do not change when the model does:

In [7]:
models = {
    "linear": LinearRegression(),
    "tree": DecisionTreeRegressor(max_depth=6, random_state=0),
    "forest": RandomForestRegressor(random_state=0),
}

for name, candidate in models.items():
    candidate.fit(X_train, y_train)
    print(f"{name:7} test R² {candidate.score(X_test, y_test):.3f}")

linear  test R² 0.888
tree    test R² 0.811
forest  test R² 0.894


Every model was trained on the same 900 stations and scored on the same 300, so the scores can be
compared. The forest's 0.894 and the linear model's 0.888 are too close for one split to tell apart,
and the **Train and Test** notebook shows a single model's score moving by more than that when only
the seed changes. The tree, limited to six levels of questions, is behind at 0.811. In **The
Estimator API** notebook the forest scored 0.984 on the 1,200 stations it had learned from, so most
of the distance between that and 0.894 is what it had memorized. A choice between models made on
these scores would use up the test set, and the **Train and Test** notebook keeps it for the end
with a validation set.

### What score means for a classifier

For a classifier, `score` is accuracy. Here the question is snow on 1 May:

In [8]:
snow = network["snow_on_may_1"]
Xs_train, Xs_test, snow_train, snow_test = train_test_split(X, snow, test_size=0.25, random_state=0)

classifier = DecisionTreeClassifier(max_depth=3, random_state=0).fit(Xs_train, snow_train)
right = (classifier.predict(Xs_test) == snow_test).sum()

print("accuracy:", round(classifier.score(Xs_test, snow_test), 3))
print(right, "of", len(snow_test), "test stations right")

accuracy: 0.747
224 of 300 test stations right


224 of the 300 test stations are right, an accuracy of 0.747. An accuracy needs something to compare
with. Snow on 1 May is the less common answer, so a rule that always says no snow would also score
well above zero:

In [9]:
print("share of test stations with no snow:", round(1 - snow_test.mean(), 3))

share of test stations with no snow: 0.593


Always guessing no snow gets 0.593 of the test stations right, so the tree's 0.747 is a real gain,
if a modest one. The **Classification Metrics** and **Imbalanced Classes** notebooks show what
accuracy leaves out.

## Your turn

Five tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than
getting there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/scikit-learn-deep-dive/02-your-first-model-solutions.ipynb).

**1.** Split `X` and `y` with `test_size=0.2` and `random_state=1`, and print how many stations went into each part.

In [10]:
# your code here

**2.** Train a `LinearRegression` on that split, and print its R² on the test set rounded to three places.

In [11]:
# your code here

**3.** Print the predicted and measured January means of the first five test stations from that model, side by side in a DataFrame.

In [12]:
# your code here

**4.** Fit `DecisionTreeRegressor(max_depth=d, random_state=0)` for `d` in 2, 4 and 8 on the split from the worked examples, and print each one's R² on the training set and on the test set.

In [13]:
# your code here

**5.** Train a `DecisionTreeClassifier(max_depth=3, random_state=0)` to predict `june_frost` on a split with `random_state=0`, and print its accuracy and the share of test stations with no frost.

In [14]:
# your code here

## Common errors

### ValueError: could not convert string to float: 'inland'

In [15]:
with_terrain = network.loc[X_train.index, FEATURES + ["terrain"]]

LinearRegression().fit(with_terrain, y_train)

ValueError: could not convert string to float: 'inland'

`terrain` holds text, and a linear model computes with numbers only, so `fit` tried to turn
`"inland"` into a float and failed. A text column has to be encoded as numbers first, which the
**Preprocessing** notebook does with `OneHotEncoder`.

### ValueError: Found input variables with inconsistent numbers of samples: [900, 300]

In [16]:
X_train_wrong, y_train_wrong, X_test_wrong, y_test_wrong = train_test_split(X, y, test_size=0.25,
                                                                            random_state=0)

LinearRegression().fit(X_train_wrong, y_train_wrong)

ValueError: Found input variables with inconsistent numbers of samples: [900, 300]

`train_test_split` returns the two parts of `X` first and then the two parts of `y`. Unpacked in
another order, the name `y_train_wrong` holds the 300 test rows of `X`, and `fit` found 900 rows of
features and 300 answers, so unpack them in the order `X_train, X_test, y_train, y_test`.

### No error, and a perfect score on the training rows

In [17]:
deep = DecisionTreeRegressor(random_state=0).fit(X_train, y_train)

print("depth:", deep.get_depth(), "| leaves:", deep.get_n_leaves())
print("R² on the training set:", round(deep.score(X_train, y_train), 3))
print("R² on the test set:    ", round(deep.score(X_test, y_test), 3))

depth: 21 | leaves: 846
R² on the training set: 1.0
R² on the test set:     0.769


With no limit on its depth, the tree kept asking questions until almost every training station sat
in a leaf of its own, 846 leaves for 900 stations, so it predicts each training station's own
January mean and scores 1.0 on them. On the test stations it scores 0.769, below the six-level tree
in the worked examples. A training score this high says the model memorized, and only the test score
says how it will do on a new station.

## Recap

- Hold back a test set with `train_test_split` before training, and unpack its four pieces in order
- Train with `fit` on the training set, and judge the model with `score` on the test set
- Read `score` as R² for a regressor and as accuracy for a classifier
- Compare models only on the same held-out rows, and compare accuracy with always guessing the more
  common answer
- Treat a training score far above the test score as memorizing

## What is next

The **Train and Test** notebook looks closer at the split itself: the seed, the size of the test
set, what happens when the rows are in order, and keeping the rare answers in both parts.

---

&#8592; **Previous:** [The Estimator API](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/scikit-learn-deep-dive/01-the-estimator-api.ipynb)  &nbsp;·&nbsp;  [scikit-learn, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/scikit-learn-deep-dive.html)  &nbsp;·&nbsp;  **Next:** [Train and Test](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/scikit-learn-deep-dive/03-train-and-test.ipynb) &#8594;
